In [1]:
{
 "cells": [
  {
   "cell_type": "code",
   "execution_count": 2,
   "metadata": {
    "colab": {
     "base_uri": "https://localhost:8080/"
    },
    "executionInfo": {
     "elapsed": 6611,
     "status": "ok",
     "timestamp": 1733393589489,
     "user": {
      "displayName": "Nicolas Bonneel",
      "userId": "01688940306486951631"
     },
     "user_tz": -60
    },
    "id": "Byegk4hOCQyY",
    "outputId": "70e3c9b0-29b2-4584-99ca-1a7e6373bec3"
   },
   "outputs": [
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "--2024-12-05 10:13:02--  https://perso.liris.cnrs.fr/nicolas.bonneel/cours/unets.py\n",
      "Resolving perso.liris.cnrs.fr (perso.liris.cnrs.fr)... 134.214.142.28\n",
      "Connecting to perso.liris.cnrs.fr (perso.liris.cnrs.fr)|134.214.142.28|:443... connected.\n",
      "HTTP request sent, awaiting response... 200 OK\n",
      "Length: 31090 (30K) [text/x-python]\n",
      "Saving to: â€˜unets.py.1â€™\n",
      "\n",
      "unets.py.1          100%[===================>]  30.36K  --.-KB/s    in 0.1s    \n",
      "\n",
      "2024-12-05 10:13:03 (213 KB/s) - â€˜unets.py.1â€™ saved [31090/31090]\n",
      "\n",
      "--2024-12-05 10:13:03--  https://perso.liris.cnrs.fr/nicolas.bonneel/cours/fullmodel_unetsmall_faces.pt\n",
      "Resolving perso.liris.cnrs.fr (perso.liris.cnrs.fr)... 134.214.142.28\n",
      "Connecting to perso.liris.cnrs.fr (perso.liris.cnrs.fr)|134.214.142.28|:443... connected.\n",
      "HTTP request sent, awaiting response... 200 OK\n",
      "Length: 77294083 (74M)\n",
      "Saving to: â€˜fullmodel_unetsmall_faces.pt.1â€™\n",
      "\n",
      "fullmodel_unetsmall 100%[===================>]  73.71M  21.6MB/s    in 4.2s    \n",
      "\n",
      "2024-12-05 10:13:08 (17.4 MB/s) - â€˜fullmodel_unetsmall_faces.pt.1â€™ saved [77294083/77294083]\n",
      "\n",
      "cuda:0\n",
      "todo\n"
     ]
    }
   ],
   "source": [
    "# please comment out once downloaded once to avoid re-downloading at each run !\n",
    "!wget https://perso.liris.cnrs.fr/nicolas.bonneel/diffusion/unets.py\n",
    "!wget https://perso.liris.cnrs.fr/nicolas.bonneel/diffusion/fullmodel_unetsmall_faces.pt\n",
    "#!wget https://perso.liris.cnrs.fr/nicolas.bonneel/diffusion/fullmodel_unetsmall_faces2.pt\n",
    "#!wget https://perso.liris.cnrs.fr/nicolas.bonneel/diffusion/fullmodel_unetsmall_photos.pt\n",
    "#!wget https://perso.liris.cnrs.fr/nicolas.bonneel/diffusion/fullmodel_unet_faces.pt\n",
    "\n",
    "\n",
    "import torch\n",
    "import torch.nn as nn\n",
    "import numpy as np\n",
    "from torchvision import datasets\n",
    "from torchvision import transforms\n",
    "import matplotlib.pyplot as plt\n",
    "import torch.nn.functional as F\n",
    "import random\n",
    "import time\n",
    "from mpl_toolkits.axes_grid1 import ImageGrid\n",
    "import unets\n",
    "\n",
    "t_max = 500  # 1000 for unet_faces, 500 for unetsmall_faces, or 200 for the photo dataset\n",
    "\n",
    "device = torch.device(\"cuda:0\" if torch.cuda.is_available() else \"cpu\")\n",
    "print(device)\n",
    "\n",
    "model = unets.UNetSmall(image_size=128, in_channels=3, out_channels=3, num_classes=None).to(device)\n",
    "\n",
    "alpha_t_vec = 1-np.linspace(0.0001, 0.02, t_max)\n",
    "alpha_bar_t_vec = np.cumprod(alpha_t_vec)\n",
    "\n",
    "checkpoint = torch.load(\"./fullmodel_unetsmall_faces.pt\", weights_only=True)\n",
    "model.load_state_dict(checkpoint['model_state_dict'])\n",
    "\n",
    "training = False\n",
    "\n",
    "\n",
    "if training:\n",
    "    resize_transform = transforms.Compose([ transforms.Resize(size=(128,128)), transforms.PILToTensor(), transforms.ConvertImageDtype(torch.float32) ]);\n",
    "\n",
    "    dataset = datasets.ImageFolder(root = \"./data\",\n",
    "                             transform = resize_transform)\n",
    "\n",
    "    loader = torch.utils.data.DataLoader(dataset = dataset,\n",
    "                                        batch_size = 1,\n",
    "                                        shuffle = True)\n",
    "    loss_function = torch.nn.MSELoss()\n",
    "\n",
    "    optimizer = torch.optim.Adam(model.parameters(), lr = 1e-4)\n",
    "\n",
    "    prefetched_images = []\n",
    "    for (image, _) in loader:\n",
    "        prefetched_images.append(np.squeeze(image*2.0-1.0))\n",
    "\n",
    "    epochs =  int(2505*12)\n",
    "    outputs = []\n",
    "    losses = []\n",
    "    batch_size = 8\n",
    "\n",
    "    start_time = time.time()\n",
    "    for epoch in range(epochs):\n",
    "        random.shuffle(prefetched_images)\n",
    "\n",
    "        #for i in range(0, int(4936/batch_size) ):\n",
    "        for i in range(0, int(48/batch_size) ):\n",
    "            batch = prefetched_images[(i*batch_size):(i*batch_size+batch_size)]\n",
    "            image = torch.stack(batch, dim=0)\n",
    "            img = image.to(device)\n",
    "            t = np.random.random_integers(0, t_max-1, size=(batch_size))\n",
    "\n",
    "            eps_0_t = torch.randn(img.size(), device=device)\n",
    "            alpha_bar_t = torch.from_numpy(alpha_bar_t_vec[t]).to(device=device).view((batch_size, 1, 1, 1)).float()\n",
    "            noisy_img = torch.sqrt(alpha_bar_t) * img + torch.sqrt(1-alpha_bar_t)*eps_0_t\n",
    "\n",
    "            predicted_noise_value = model(noisy_img, torch.from_numpy(t).float().to(device=device))\n",
    "\n",
    "            loss = loss_function(predicted_noise_value, eps_0_t)\n",
    "\n",
    "            reconstructed = (noisy_img - torch.sqrt(1-alpha_bar_t)*predicted_noise_value)/torch.sqrt(alpha_bar_t)\n",
    "\n",
    "            optimizer.zero_grad()\n",
    "            loss.backward()\n",
    "            optimizer.step()\n",
    "\n",
    "            losses.append(loss.cpu().detach())\n",
    "\n",
    "    print(\"--- %s seconds ---\" % (time.time() - start_time))\n",
    "\n",
    "    torch.save({\n",
    "                'epoch': epoch,\n",
    "                'model_state_dict': model.state_dict(),\n",
    "                'optimizer_state_dict': optimizer.state_dict(),\n",
    "                'loss': loss,\n",
    "                }, \"./fullmodel.pt\")\n",
    "\n",
    "\n",
    "    plt.figure(0)\n",
    "\n",
    "    plt.style.use('fivethirtyeight')\n",
    "    plt.xlabel('Iterations')\n",
    "    plt.ylabel('Loss')\n",
    "\n",
    "    plt.plot(losses[10:])\n",
    "\n",
    "    # Show noisy images\n",
    "    fig = plt.figure(1, figsize=(6, 6))\n",
    "    grid = ImageGrid(fig, 111,\n",
    "                 nrows_ncols=(4, 4),\n",
    "                 axes_pad=0.01,\n",
    "                 )\n",
    "    for i, (ax, item)  in enumerate(zip(grid, noisy_img.cpu().detach())):\n",
    "        if i==16:\n",
    "            break\n",
    "        item = (np.transpose(item.reshape(3, 128, 128), (1,2,0))+1.0)/2.0\n",
    "        ax.imshow(item)\n",
    "\n",
    "    # Show denoised images\n",
    "    fig = plt.figure(2, figsize=(6, 6))\n",
    "    grid = ImageGrid(fig, 111,\n",
    "                 nrows_ncols=(4, 4),\n",
    "                 axes_pad=0.01,\n",
    "                 )\n",
    "\n",
    "    for i, (ax, item) in enumerate(zip(grid , reconstructed.cpu().detach())):\n",
    "        if i==16:\n",
    "            break\n",
    "        item = (np.transpose(item.reshape(3, 128, 128),(1,2,0))+1.0)/2.0\n",
    "        ax.imshow(item)\n",
    "\n",
    "\n",
    "### the denoiser has been learnt.\n",
    "\n",
    "### Now generate an image\n",
    "\n",
    "noisy_stuff = torch.randn((1,3,128,128), device=device)\n",
    "\n",
    "model.eval()\n",
    "with torch.no_grad():\n",
    "  # loop over time and progressively denoise your randn\n",
    "  print(\"todo\")\n"
   ]
  }
 ],
 "metadata": {
  "accelerator": "GPU",
  "colab": {
   "authorship_tag": "ABX9TyNWyZmjxpQYasyoxy/Hlrj1",
   "gpuType": "T4",
   "provenance": []
  },
  "kernelspec": {
   "display_name": "Python 3 (ipykernel)",
   "language": "python",
   "name": "python3"
  },
  "language_info": {
   "codemirror_mode": {
    "name": "ipython",
    "version": 3
   },
   "file_extension": ".py",
   "mimetype": "text/x-python",
   "name": "python",
   "nbconvert_exporter": "python",
   "pygments_lexer": "ipython3",
   "version": "3.13.2"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 4
}

{'cells': [{'cell_type': 'code',
   'execution_count': 2,
   'metadata': {'colab': {'base_uri': 'https://localhost:8080/'},
    'executionInfo': {'elapsed': 6611,
     'status': 'ok',
     'timestamp': 1733393589489,
     'user': {'displayName': 'Nicolas Bonneel',
      'userId': '01688940306486951631'},
     'user_tz': -60},
    'id': 'Byegk4hOCQyY',
    'outputId': '70e3c9b0-29b2-4584-99ca-1a7e6373bec3'},
   'outputs': [{'name': 'stdout',
     'output_type': 'stream',
     'text': ['--2024-12-05 10:13:02--  https://perso.liris.cnrs.fr/nicolas.bonneel/cours/unets.py\n',
      'Resolving perso.liris.cnrs.fr (perso.liris.cnrs.fr)... 134.214.142.28\n',
      'Connecting to perso.liris.cnrs.fr (perso.liris.cnrs.fr)|134.214.142.28|:443... connected.\n',
      'HTTP request sent, awaiting response... 200 OK\n',
      'Length: 31090 (30K) [text/x-python]\n',
      'Saving to: â€˜unets.py.1â€™\n',
      '\n',
      'unets.py.1          100%[===================>]  30.36K  --.-KB/s    in 0.1s  